In [1]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Cell 1 done — Dependencies installed')

✅ Cell 1 done — Dependencies installed


In [2]:
# =====================================================================
# CELL 2: Imports
# =====================================================================

import os, re, json, torch
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.makedirs('results', exist_ok=True)
print('✅ Cell 2 done — Imports ready')

✅ Cell 2 done — Imports ready


In [2]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00


In [3]:
# =====================================================================
# CELL 3: Upload & Load the FinanceBench Data
# =====================================================================

# OPTION A: If files are already in Colab (uploaded previously)
# Just make sure financebench_open_source.jsonl is in your working directory

# OPTION B: Upload now
# from google.colab import files
# uploaded = files.upload()  # upload financebench_open_source.jsonl

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

qa = load_jsonl('financebench_open_source.jsonl')

evidence_map = {}
for item in qa:
    evidence_map[item['financebench_id']] = item.get('evidence', [])

print(f'✅ Cell 3 done — Loaded {len(qa)} questions')
print(f'\nFirst 5 questions and answers:')
for i, item in enumerate(qa[:5]):
    print(f'  Q{i+1}: {item["question"][:65]}...')
    print(f'  A{i+1}: {item["answer"][:65]}')
    print()


✅ Cell 3 done — Loaded 150 questions

First 5 questions and answers:
  Q1: What is the FY2018 capital expenditure amount (in USD millions) f...
  A1: $1577.00

  Q2: Assume that you are a public equities analyst. Answer the followi...
  A2: $8.70

  Q3: Is 3M a capital-intensive business based on FY2022 data?...
  A3: No, the company is managing its CAPEX and Fixed Assets pretty eff

  Q4: What drove operating margin change as of FY2022 for 3M? If operat...
  A4: Operating Margin for 3M in FY2022 has decreased by 1.7% primarily

  Q5: If we exclude the impact of M&A, which segment has dragged down 3...
  A5: The consumer segment shrunk by 0.9% organically.



In [4]:
# ================== CELL 4: Load Llama 3.1 8B ==================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.1f} GB')

MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'

print(f'\nLoading {MODEL_NAME} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

mem_used = torch.cuda.memory_allocated() / 1e9
print(f'\n Llama 3.1 8B loaded!')
print(f'   GPU memory: {mem_used:.1f} GB / {mem_gb:.1f} GB')

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


✅ Llama 3.1 8B loaded!
   GPU memory: 6.1 GB / 42.4 GB


In [5]:
# Delete old model
import gc
del model
gc.collect()
torch.cuda.empty_cache()

# Reload in float16 (no quantization)
print("Reloading Llama 3.1 8B in float16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

mem_used = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory: {mem_used:.1f} GB')

# Test immediately
messages = [{"role": "user", "content": "What is 2+2? Answer with just the number."}]
formatted = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(formatted, return_tensors='pt', truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False,
                         pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
new_tokens = out[0][inputs['input_ids'].shape[1]:]
result = tok.decode(new_tokens, skip_special_tokens=True).strip()
print(f'Test: "{result[:100]}"')

Reloading Llama 3.1 8B in float16...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GPU memory: 16.1 GB
Test: "4"


In [6]:
# =====================================================================
# CELL 5: Define All Functions
# =====================================================================

import re, json, csv
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def extract_all_numbers(text):
    if not text:
        return []
    cleaned = text.replace('$', '').replace('%', '')

    numbers = []
    used_positions = set()

    # Step 1: Parenthesized negatives
    for m in re.finditer(r'\((\d[\d,]*(?:\.\d+)?)\)', cleaned):
        num_str = m.group(1).replace(',', '')
        try:
            numbers.append(-float(num_str))
            for pos in range(m.start(), m.end()):
                used_positions.add(pos)
        except:
            pass

    # Step 2: All other numbers: use a pattern that grabs the full number
    for m in re.finditer(r'-?\d[\d,]*\.?\d*', cleaned):
        if any(pos in used_positions for pos in range(m.start(), m.end())):
            continue
        num_str = m.group(0).replace(',', '')
        try:
            val = float(num_str)
            numbers.append(val)
        except:
            pass

    return numbers

def extract_first_number(text):
    nums = extract_all_numbers(text)
    return nums[0] if nums else None

def normalize_value_to_base(value, text):
    text_lower = text.lower()
    if re.search(r'\bbillion\b', text_lower) or re.search(r'\d\s*b\b', text_lower):
        return value * 1_000_000_000
    if re.search(r'\bmillion\b', text_lower) or re.search(r'\d\s*m\b', text_lower):
        return value * 1_000_000
    if re.search(r'\bthousand\b', text_lower) or re.search(r'\d\s*k\b', text_lower):
        return value * 1_000
    return value

def numerical_accuracy(pred, gold):
    gold_nums = extract_all_numbers(gold)
    pred_nums = extract_all_numbers(pred)
    if not gold_nums:
        return 1.0 if not pred_nums else 0.0
    if not pred_nums:
        return 0.0
    for g in gold_nums:
        matched = False
        for p in pred_nums:
            if g == 0:
                if abs(p) < 1e-6: matched = True; break
            elif abs(p - g) / abs(g) < 0.05:
                matched = True; break
        if not matched:
            g_base = normalize_value_to_base(g, gold)
            for p in pred_nums:
                p_base = normalize_value_to_base(p, pred)
                if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
                    matched = True; break
            if not matched:
                return 0.0
    return 1.0

def numerical_hallucination_rate(pred, evidence_list):
    if not pred:
        return 0.0
    pred_nums = extract_all_numbers(pred)
    if not pred_nums:
        return 0.0
    evidence_nums = set()
    for ev in evidence_list:
        if isinstance(ev, dict):
            ev_text = ev.get('evidence_text', '') + ' ' + ev.get('evidence_text_full_page', '')
        else:
            ev_text = str(ev)
        for n in extract_all_numbers(ev_text):
            evidence_nums.add(abs(n))
    if not evidence_nums:
        return 1.0
    hallucinated = 0
    for p in pred_nums:
        p_abs = abs(p)
        found = False
        for e in evidence_nums:
            if e == 0:
                if p_abs < 1e-6: found = True; break
            elif abs(p_abs - e) / max(abs(e), 1e-10) < 0.05:
                found = True; break
        if not found:
            hallucinated += 1
    return hallucinated / len(pred_nums)

def calculate_bleu(reference, candidate):
    if not reference or not candidate:
        return 0.0
    ref_tokens = [reference.lower().split()]
    cand_tokens = candidate.lower().split()
    if not cand_tokens:
        return 0.0
    smoothing = SmoothingFunction().method1
    try:
        return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothing)
    except:
        return 0.0

def has_citation(candidate):
    if not candidate:
        return 0.0
    candidate_lower = candidate.lower()
    patterns = [
        r'according to', r'based on',
        r'as (stated|reported|shown|noted|mentioned) in',
        r'per the', r'page \d+',
        r'10-k\b', r'10k\b', r'10-q\b',
        r'\bfiling\b', r'\bannual report\b',
        r'financial statement', r'cash flow statement',
        r'income statement', r'balance sheet',
        r'\bsec\b', r'\b(form|item)\s+\d',
        r'source:', r'\[.*?\]', r'evidence',
    ]
    return 1.0 if any(re.search(p, candidate_lower) for p in patterns) else 0.0

def normalize(s):
    return re.sub(r'\s+', ' ', s.strip().lower())

def is_correct(pred, gold):
    if normalize(pred) == normalize(gold):
        return True
    p = extract_first_number(pred)
    g = extract_first_number(gold)
    if p is not None and g is not None:
        if g != 0 and abs(p - g) / abs(g) < 0.05:
            return True
        p_base = normalize_value_to_base(p, pred)
        g_base = normalize_value_to_base(g, gold)
        if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
            return True
    return False

def build_base_prompt(question):
    return f"""You are a financial analyst answering questions about public company 10-K filings.
Answer with ONLY the specific number or value asked for.
Do NOT explain. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%" or "Yes" or "No")

Question: {question}

Answer:"""

# def generate_answer(prompt, max_new_tokens=64):
#     inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=2048)
#     # Send to the model's device AND dtype
#     inputs = {k: v.to(model.device) for k, v in inputs.items()}
#     with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
#         out = model.generate(
#             **inputs,
#             max_new_tokens=64,
#             do_sample=False,
#             pad_token_id=tok.pad_token_id,
#             eos_token_id=tok.eos_token_id,
#         )
#     new_tokens = out[0][inputs['input_ids'].shape[1]:]
#     raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
#     return raw.split('\n')[0].strip()

# print('✅ generate_answer fixed with torch.autocast bfloat16')

def extract_answer_from_verbose(pred):
    if not pred:
        return pred
    dollar = re.search(r'\$\s*[\d,]+(?:\.\d+)?\s*(?:billion|million|B|M|K)?', pred, re.IGNORECASE)
    if dollar:
        return dollar.group(0).strip()
    pct = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*%', pred)
    if pct:
        return pct.group(0).strip()
    with_units = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*(?:billion|million|thousand|B|M|K)', pred, re.IGNORECASE)
    if with_units:
        return with_units.group(0).strip()
    plain = re.search(r'[-+]?\$?\s*[\d,]+(?:\.\d+)?', pred)
    if plain:
        return plain.group(0).strip()
    if len(pred.split()) < 15:
        return pred
    return pred

print('✅ Cell 5 done — All functions defined')

✅ Cell 5 done — All functions defined


In [7]:
# =====================================================================
# CELL 5b: Fix dtype — cast each parameter individually
# =====================================================================

# Cast every parameter to bfloat16 in-place (works with offloaded models)
for name, param in model.named_parameters():
    if param.dtype != torch.bfloat16:
        param.data = param.data.to(torch.bfloat16)

for name, buf in model.named_buffers():
    if buf.dtype.is_floating_point and buf.dtype != torch.bfloat16:
        buf.data = buf.data.to(torch.bfloat16)

print('✅ All parameters cast to bfloat16')

# =====================================================================
# CELL 5b: Generate function for Llama
# =====================================================================

# =====================================================================
# CELL 5b: Generate function for Llama (with chat template)
# =====================================================================

def generate_answer(prompt, max_new_tokens=64):
    # Use Llama's chat template
    messages = [
        {"role": "user", "content": prompt}
    ]
    input_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(input_text, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return raw.split('\n')[0].strip()

# Smoke test
test = generate_answer('What is 2+2? Answer with just the number:')
print(f'Smoke test: "{test}"')
print('✅ Generation working!')

✅ All parameters cast to bfloat16
Smoke test: "4"
✅ Generation working!


In [8]:
!pip install -q rank_bm25
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.8 MB/s eta 0:00:00


In [9]:
# =====================================================================
# CELL 9a: Load RAG System — BM25
# =====================================================================
from rank_bm25 import BM25Okapi
import os, re, pickle, shutil
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

os.makedirs('rag/vector_index', exist_ok=True)

if not os.path.exists('rag/vector_index/faiss_index.bin'):
    print('Upload faiss_index.bin and evidence_store.pkl:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if 'faiss' in fname:
            with open('rag/vector_index/faiss_index.bin', 'wb') as f:
                f.write(uploaded[fname])
        if 'evidence' in fname or 'pkl' in fname:
            with open('rag/vector_index/evidence_store.pkl', 'wb') as f:
                f.write(uploaded[fname])
    print('✅ Files uploaded')
else:
    print('✅ RAG files already exist')

# ─── Company aliases ─────────────────────────────────────────

COMPANY_ALIASES = {
    '3m': '3M', 'mmm': '3M',
    'aes': 'AES Corporation', 'aes corporation': 'AES Corporation',
    'amcor': 'Amcor',
    'amd': 'AMD', 'advanced micro': 'AMD',
    'activision': 'Activision Blizzard', 'activision blizzard': 'Activision Blizzard',
    'adobe': 'Adobe',
    'american express': 'American Express', 'amex': 'American Express',
    'american water': 'American Water Works', 'american water works': 'American Water Works',
    'best buy': 'Best Buy',
    'boeing': 'Boeing',
    'cvs': 'CVS Health', 'cvs health': 'CVS Health',
    'coca-cola': 'Coca-Cola', 'coca cola': 'Coca-Cola', 'coke': 'Coca-Cola',
    'corning': 'Corning',
    'costco': 'Costco',
    'foot locker': 'Foot Locker',
    'general mills': 'General Mills',
    'jpmorgan': 'JPMorgan', 'jpm': 'JPMorgan', 'jp morgan': 'JPMorgan',
    'johnson': 'Johnson & Johnson', 'johnson & johnson': 'Johnson & Johnson', 'j&j': 'Johnson & Johnson',
    'lockheed': 'Lockheed Martin', 'lockheed martin': 'Lockheed Martin',
    'mgm': 'MGM Resorts', 'mgm resorts': 'MGM Resorts',
    'microsoft': 'Microsoft', 'msft': 'Microsoft',
    'nike': 'Nike',
    'netflix': 'Netflix', 'nflx': 'Netflix',
    'paypal': 'Paypal',
    'pepsico': 'PepsiCo', 'pepsi': 'PepsiCo',
    'pfizer': 'Pfizer',
    'ulta': 'Ulta Beauty', 'ulta beauty': 'Ulta Beauty',
    'verizon': 'Verizon',
    'walmart': 'Walmart',
    'block': 'Block', 'square': 'Block',
    'amazon': 'Amazon', 'amzn': 'Amazon',
    'kraft': 'Kraft Heinz', 'kraft heinz': 'Kraft Heinz',
}

def extract_company(question):
    q_lower = question.lower()
    for alias in sorted(COMPANY_ALIASES.keys(), key=len, reverse=True):
        if alias in q_lower:
            return COMPANY_ALIASES[alias]
    return None

def extract_year(question):
    fy_match = re.search(r'FY\s*(\d{4})', question, re.IGNORECASE)
    if fy_match:
        return fy_match.group(1)
    year_match = re.search(r'20\d{2}', question)
    if year_match:
        return year_match.group(0)
    return None

# ─── RAG Class with HARD filtering ──────────────────────────

class FinancialRAG:
    def __init__(self, embedding_model='all-MiniLM-L6-v2'):
        print(f"Loading embedding model: {embedding_model}")
        self.encoder = SentenceTransformer(embedding_model)
        self.evidence_store = []
        self.index = None
        self.all_embeddings = None

    def load_index(self, index_dir='./rag/vector_index'):
        self.index = faiss.read_index(f'{index_dir}/faiss_index.bin')
        with open(f'{index_dir}/evidence_store.pkl', 'rb') as f:
            self.evidence_store = pickle.load(f)

        # Extract all embeddings for filtered cosine similarity
        self.all_embeddings = np.zeros((self.index.ntotal, self.index.d), dtype='float32')
        for i in range(self.index.ntotal):
            self.all_embeddings[i] = self.index.reconstruct(i)

        print(f"✅ Index loaded: {self.index.ntotal} vectors, {len(self.evidence_store)} chunks")
        print(f"✅ Embeddings extracted for filtered search")

    def _cosine_sim(self, query_emb, doc_embs):
        query_norm = query_emb / (np.linalg.norm(query_emb) + 1e-10)
        doc_norms = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-10)
        return np.dot(doc_norms, query_norm.T).flatten()

    def retrieve(self, question, k=3):
        question_embedding = self.encoder.encode([question], convert_to_numpy=True).astype('float32')
        target_company = extract_company(question)
        target_year = extract_year(question)

        # ─── STEP 1: Hard filter — company + year ────────────
        filtered = []
        for i, (text, metadata) in enumerate(self.evidence_store):
            meta_company = metadata.get('company', '').lower()
            meta_doc = metadata.get('doc_name', '')

            company_match = target_company and target_company.lower() in meta_company
            year_match = target_year and target_year in meta_doc

            if company_match and year_match:
                filtered.append(i)

        # ─── STEP 2: If ZERO, relax to company only ──────────
        if len(filtered) == 0:
            for i, (text, metadata) in enumerate(self.evidence_store):
                meta_company = metadata.get('company', '').lower()
                if target_company and target_company.lower() in meta_company:
                    filtered.append(i)

        # ─── STEP 3: If still ZERO, search all ───────────────
        if len(filtered) == 0:
            filtered = list(range(len(self.evidence_store)))

        # ─── STEP 4: HYBRID scoring (FAISS + BM25) ───────────

        # Score A: Cosine similarity (semantic)
        filtered_embeddings = self.all_embeddings[filtered]
        cos_scores = self._cosine_sim(question_embedding[0], filtered_embeddings)

        # Normalize cosine scores to 0-1
        if cos_scores.max() != cos_scores.min():
            cos_norm = (cos_scores - cos_scores.min()) / (cos_scores.max() - cos_scores.min())
        else:
            cos_norm = np.ones_like(cos_scores)

        # Score B: BM25 (keyword matching)
        filtered_texts = [self.evidence_store[i][0] for i in filtered]
        tokenized_docs = [doc.lower().split() for doc in filtered_texts]
        bm25 = BM25Okapi(tokenized_docs)
        bm25_scores = bm25.get_scores(question.lower().split())

        # Normalize BM25 scores to 0-1
        if bm25_scores.max() != bm25_scores.min():
            bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
        else:
            bm25_norm = np.zeros_like(bm25_scores)

        # Combined score: 50% semantic + 50% keyword
        combined_scores = 0.5 * cos_norm + 0.5 * bm25_norm

        actual_k = min(k, len(filtered))
        top_indices = np.argsort(combined_scores)[::-1][:actual_k]

        candidates = []
        for idx in top_indices:
            real_idx = filtered[idx]
            text, metadata = self.evidence_store[real_idx]
            candidates.append({
                'evidence_text': text,
                'metadata': metadata,
                'distance': float(1 - combined_scores[idx]),
            })

        return candidates

# ─── Initialize ──────────────────────────────────────────────

rag = FinancialRAG()
rag.load_index('./rag/vector_index')

# ─── Test ────────────────────────────────────────────────────

print('\n' + '='*60)
print('HARD METADATA FILTERING TEST')
print('='*60)

test_qs = [
    "What is the FY2018 capital expenditure amount (in USD millions) for 3M?",
    "What is the FY2022 total revenue for Amazon?",
    "Does 3M have a reasonably healthy liquidity profile based on its FY2023 10Q?",
]

for q in test_qs:
    company = extract_company(q)
    year = extract_year(q)
    results = rag.retrieve(q, k=3)

    print(f'\nQ: {q[:70]}...')
    print(f'  Extracted → Company: {company}, Year: {year}')
    for j, r in enumerate(results):
        m = r['metadata']
        print(f'  [{j+1}] {m["company"]} - {m["doc_name"]} (score: {r["distance"]:.4f})')

print('\n✅ Cell 9a done — Hard metadata filtering RAG loaded!')

Upload faiss_index.bin and evidence_store.pkl:


Saving faiss_index.bin to faiss_index.bin
Saving evidence_store.pkl to evidence_store.pkl
✅ Files uploaded
Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Index loaded: 189 vectors, 189 chunks
✅ Embeddings extracted for filtered search

HARD METADATA FILTERING TEST

Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M...
  Extracted → Company: 3M, Year: 2018
  [1] 3M - 3M_2018_10K (score: 0.5000)
  [2] 3M - 3M_2018_10K (score: 0.5000)

Q: What is the FY2022 total revenue for Amazon?...
  Extracted → Company: Amazon, Year: 2022
  [1] Amazon - AMAZON_2017_10K (score: 0.5000)
  [2] Amazon - AMAZON_2017_10K (score: 0.5000)
  [3] Amazon - AMAZON_2017_10K (score: 0.5000)

Q: Does 3M have a reasonably healthy liquidity profile based on its FY202...
  Extracted → Company: 3M, Year: 2023
  [1] 3M - 3M_2023Q2_10Q (score: 0.2376)
  [2] 3M - 3M_2023Q2_10Q (score: 0.5000)
  [3] 3M - 3M_2023Q2_10Q (score: 0.5000)

✅ Cell 9a done — Hard metadata filtering RAG loaded!


In [10]:
# =====================================================================
# CELL 9b: Llama 3.1 8B +  BM25 — Split by Question Type -> NEW WITH with hard metadata filtering.
# =====================================================================

# ─── Step 1: Categorize all questions ────────────────────────

def categorize_question(gold_answer):
    gold_clean = gold_answer.strip().lower()
    if gold_clean.startswith('yes') or gold_clean.startswith('no'):
        return 'yes_no'
    words = gold_answer.strip().split()
    if len(words) <= 4:
        nums = extract_all_numbers(gold_answer)
        if nums:
            return 'numerical'
    return 'descriptive'

for ex in qa:
    ex['category'] = categorize_question(ex['answer'])

cat_counts = {}
for ex in qa:
    cat_counts[ex['category']] = cat_counts.get(ex['category'], 0) + 1

print(f'Question Categories:')
print(f'  Numerical:   {cat_counts.get("numerical", 0)}')
print(f'  Yes/No:      {cat_counts.get("yes_no", 0)}')
print(f'  Descriptive: {cat_counts.get("descriptive", 0)}')
print(f'  Total:       {len(qa)}')

# ─── Step 2: Different prompts per category ──────────────────

def build_rag_prompt_numerical(question, evidence_text):
    return f"""You are a financial data extraction tool.
Extract ONLY the specific number or value asked for from the evidence below.
Do NOT explain. Do NOT repeat the question. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%")

Evidence:
{evidence_text}

Question: {question}

Answer (number only):"""

def build_rag_prompt_yesno(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Start with "Yes" or "No", then give a brief explanation with key numbers from the evidence.

Evidence:
{evidence_text}

Question: {question}

Answer (start with Yes or No):"""

def build_rag_prompt_descriptive(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Give a concise answer using specific numbers and facts from the evidence.
Cite the source document when possible.

Evidence:
{evidence_text}

Question: {question}

Answer:"""

# ─── Step 3: Run evaluation ─────────────────────────────────

print(f'\n{"=" * 70}')
print(f'Llama 3.1 8B +  BM25 — {len(qa)} Questions')
print(f'{"=" * 70}')

correct_rag = 0
results_rag = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence = ex.get('evidence', [])
    category = ex['category']

    # RAG: Retrieve evidence
    retrieved = rag.retrieve(q, k=3)
    evidence_text = '\n\n'.join([r['evidence_text'][:800] for r in retrieved])

    # Pick the right prompt for this question type
    if category == 'numerical':
        prompt = build_rag_prompt_numerical(q, evidence_text)
    elif category == 'yes_no':
        prompt = build_rag_prompt_yesno(q, evidence_text)
    else:
        prompt = build_rag_prompt_descriptive(q, evidence_text)

    # Generate
    try:
        pred_raw = generate_answer(prompt)
        pred = extract_answer_from_verbose(pred_raw) if category == 'numerical' else pred_raw
    except Exception as e:
        print(f'  ⚠️ Error Q{i+1}: {e}')
        pred_raw = pred = 'ERROR'

    # Correctness check
    ok = is_correct(pred, gold)
    correct_rag += int(ok)

    # Store all info for split scoring later
    results_rag.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': pred_raw,
        'pred_answer': pred,
        'correct': ok,
        'category': category,
    })

    if (i + 1) % 10 == 0:
        acc_so_far = correct_rag / (i + 1) * 100
        print(f'  [{i+1:>3}/{len(qa)}] Accuracy: {acc_so_far:.1f}%')

    # Show first 2 of each category
    shown = {'numerical': 0, 'yes_no': 0, 'descriptive': 0}
    for r in results_rag:
        if r['category'] == category and shown[category] < 2:
            if i < 6:
                print(f'\n  [{category.upper()}] Q: {q[:65]}...')
                print(f'    Gold: {gold[:70]}')
                print(f'    Pred: {pred_raw[:70]}')
                print(f'    Correct: {"✅" if ok else "❌"}')
            shown[category] += 1

print(f'\n  Overall Accuracy: {correct_rag}/{len(qa)} ({correct_rag/len(qa)*100:.1f}%)')


# ─── Step 4: Split Scoring ───────────────────────────────────

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# NUMERICAL
num_r = [r for r in results_rag if r['category'] == 'numerical']
num_acc_list = [numerical_accuracy(r['pred_answer'], r['gold_answer']) for r in num_r]
num_hal_list = [numerical_hallucination_rate(r['pred_answer'], evidence_map.get(r['financebench_id'], [])) for r in num_r]
num_cit_list = [has_citation(r['pred_raw']) for r in num_r]

# YES/NO
yn_r = [r for r in results_rag if r['category'] == 'yes_no']
yn_correct = 0
yn_bleu_list = []
yn_sem_list = []
yn_cit_list = []
for r in yn_r:
    g = r['gold_answer'].strip().lower()
    p = r['pred_answer'].strip().lower()
    if (g.startswith('yes') and p.startswith('yes')) or (g.startswith('no') and p.startswith('no')):
        yn_correct += 1
    yn_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    yn_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    yn_cit_list.append(has_citation(r['pred_raw']))

# DESCRIPTIVE
desc_r = [r for r in results_rag if r['category'] == 'descriptive']
desc_bleu_list = []
desc_sem_list = []
desc_keynum_list = []
desc_cit_list = []
for r in desc_r:
    desc_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    desc_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    gold_nums = extract_all_numbers(r['gold_answer'])
    pred_nums = extract_all_numbers(r['pred_answer'])
    if gold_nums:
        matched = 0
        for g in gold_nums:
            for p in pred_nums:
                if g != 0 and abs(p - g) / abs(g) < 0.05:
                    matched += 1; break
                elif g == 0 and abs(p) < 1e-6:
                    matched += 1; break
        desc_keynum_list.append(matched / len(gold_nums))
    else:
        desc_keynum_list.append(1.0)
    desc_cit_list.append(has_citation(r['pred_raw']))


# ─── Step 5: Print Results ───────────────────────────────────

print(f'\n{"=" * 70}')
print(f'  Llama 3.1 8B +  BM25 — SPLIT RESULTS')
print(f'{"=" * 70}')

print(f'\n  📊 NUMERICAL ({len(num_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Numerical Accuracy:      {np.mean(num_acc_list)*100:.1f}%  ({int(sum(num_acc_list))}/{len(num_r)})')
print(f'  Numerical Hallucination: {np.mean(num_hal_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(num_cit_list)*100:.1f}%')

print(f'\n  ✅❌ YES/NO ({len(yn_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Yes/No Accuracy:         {yn_correct/len(yn_r)*100:.1f}%  ({yn_correct}/{len(yn_r)})')
print(f'  BLEU Score:              {np.mean(yn_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(yn_sem_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(yn_cit_list)*100:.1f}%')

print(f'\n  📝 DESCRIPTIVE ({len(desc_r)} questions)')
print(f'  {"─" * 50}')
print(f'  BLEU Score:              {np.mean(desc_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(desc_sem_list)*100:.1f}%')
print(f'  Key Number Capture:      {np.mean(desc_keynum_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(desc_cit_list)*100:.1f}%')

print(f'\n{"=" * 70}')

# Save
with open('results/llama_8b_rag_detailed_BM25.json', 'w') as f:
    json.dump(results_rag, f, indent=2)

with open('results/llama_8b_rag_split_metrics_BM25.json', 'w') as f:
    json.dump({
        'model': 'Llama-3.1-8B + RAG',
        'numerical': {
            'count': len(num_r),
            'accuracy': round(np.mean(num_acc_list)*100, 1),
            'hallucination': round(np.mean(num_hal_list)*100, 1),
            'citation': round(np.mean(num_cit_list)*100, 1),
        },
        'yes_no': {
            'count': len(yn_r),
            'yn_accuracy': round(yn_correct/len(yn_r)*100, 1),
            'bleu': round(np.mean(yn_bleu_list), 3),
            'semantic_similarity': round(np.mean(yn_sem_list)*100, 1),
            'citation': round(np.mean(yn_cit_list)*100, 1),
        },
        'descriptive': {
            'count': len(desc_r),
            'bleu': round(np.mean(desc_bleu_list), 3),
            'semantic_similarity': round(np.mean(desc_sem_list)*100, 1),
            'key_number_capture': round(np.mean(desc_keynum_list)*100, 1),
            'citation': round(np.mean(desc_cit_list)*100, 1),
        }
    }, f, indent=2)

print(f'\n✅ Cell 9b done!')
print(f'📌 Next: Run Cell 10 for BASE vs RAG comparison')

Question Categories:
  Numerical:   53
  Yes/No:      37
  Descriptive: 60
  Total:       150

Llama 3.1 8B +  BM25 — 150 Questions

  [NUMERICAL] Q: What is the FY2018 capital expenditure amount (in USD millions) f...
    Gold: $1577.00
    Pred: 1,488
    Correct: ❌

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.738B
    Correct: ✅

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.738B
    Correct: ✅

  [YES_NO] Q: Is 3M a capital-intensive business based on FY2022 data?...
    Gold: No, the company is managing its CAPEX and Fixed Assets pretty efficien
    Pred: Yes
    Correct: ❌

  [DESCRIPTIVE] Q: What drove operating margin change as of FY2022 for 3M? If operat...
    Gold: Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due 
    Pred: Based on the provided evidence, the operating margin change for 3M as 
    Correct: ✅

  [DESC

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Llama 3.1 8B +  BM25 — SPLIT RESULTS

  📊 NUMERICAL (53 questions)
  ──────────────────────────────────────────────────
  Numerical Accuracy:      20.8%  (11/53)
  Numerical Hallucination: 49.1%
  Citation Rate:           1.9%

  ✅❌ YES/NO (37 questions)
  ──────────────────────────────────────────────────
  Yes/No Accuracy:         64.9%  (24/37)
  BLEU Score:              0.000
  Semantic Similarity:     17.2%
  Citation Rate:           0.0%

  📝 DESCRIPTIVE (60 questions)
  ──────────────────────────────────────────────────
  BLEU Score:              0.060
  Semantic Similarity:     58.0%
  Key Number Capture:      63.5%
  Citation Rate:           85.0%


✅ Cell 9b done!
📌 Next: Run Cell 10 for BASE vs RAG comparison


In [11]:
# =====================================================================
# DEBUG: Check what the model sees for wrong numerical answers
# =====================================================================

wrong_numerical = [r for r in results_rag if r['category'] == 'numerical' and not r['correct']]

print(f"Wrong numerical answers: {len(wrong_numerical)}/53\n")

for i, r in enumerate(wrong_numerical[:5]):
    q = r['question']
    gold = r['gold_answer']
    pred = r['pred_answer']
    pred_raw = r['pred_raw']

    # Get the retrieved evidence
    retrieved = rag.retrieve(q, k=3)

    print(f"{'='*70}")
    print(f"Q{i+1}: {q[:80]}...")
    print(f"Gold: {gold}")
    print(f"Pred raw: {pred_raw[:150]}")
    print(f"Pred clean: {pred}")
    print(f"\nRetrieved documents:")
    for j, ret in enumerate(retrieved):
        m = ret['metadata']
        print(f"  [{j+1}] {m['company']} - {m['doc_name']}")

    print(f"\nEvidence (first 300 chars):")
    evidence_text = '\n\n'.join([ret['evidence_text'][:800] for ret in retrieved])
    print(f"  {evidence_text[:300]}...")
    print()

Wrong numerical answers: 42/53

Q1: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a r...
Gold: $1577.00
Pred raw: 1,488
Pred clean: 1,488

Retrieved documents:
  [1] 3M - 3M_2018_10K
  [2] 3M - 3M_2018_10K

Evidence (first 300 chars):
  Table of Contents 
3M Company and Subsidiaries
Consolidated Balance Shee t
At December 31
 
 
 
December 31,
 
December 31,
 
(Dollars in millions, except per share amount)
 
2018
 
2017
 
Assets
 
 
 
 
 
Current assets
 
 
 
 
 
Cash and cash equivalents
 
$
2,853 
$
3,053 
Marketable securities c...

Q2: What is the FY2019 fixed asset turnover ratio for Activision Blizzard? Fixed ass...
Gold: 24.26
Pred raw: 7.50
Pred clean: 7.50

Retrieved documents:
  [1] Activision Blizzard - ACTIVISIONBLIZZARD_2019_10K
  [2] Activision Blizzard - ACTIVISIONBLIZZARD_2019_10K
  [3] Activision Blizzard - ACTIVISIONBLIZZARD_2019_10K

Evidence (first 300 chars):
  Table of Contents
ACTIVISION BLIZZARD, INC. AND SUBSIDIARIES
CONSOLIDATED 

# ** Statement Routing**

In [12]:
# =====================================================================
# CELL 9a: Load RAG System — BM25 + Statement Type Detection
# =====================================================================
from rank_bm25 import BM25Okapi
import os, re, pickle, shutil
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

os.makedirs('rag/vector_index', exist_ok=True)

if not os.path.exists('rag/vector_index/faiss_index.bin'):
    print('Upload faiss_index.bin and evidence_store.pkl:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if 'faiss' in fname:
            with open('rag/vector_index/faiss_index.bin', 'wb') as f:
                f.write(uploaded[fname])
        if 'evidence' in fname or 'pkl' in fname:
            with open('rag/vector_index/evidence_store.pkl', 'wb') as f:
                f.write(uploaded[fname])
    print('✅ Files uploaded')
else:
    print('✅ RAG files already exist')

# ─── Company aliases ─────────────────────────────────────────

COMPANY_ALIASES = {
    '3m': '3M', 'mmm': '3M',
    'aes': 'AES Corporation', 'aes corporation': 'AES Corporation',
    'amcor': 'Amcor',
    'amd': 'AMD', 'advanced micro': 'AMD',
    'activision': 'Activision Blizzard', 'activision blizzard': 'Activision Blizzard',
    'adobe': 'Adobe',
    'american express': 'American Express', 'amex': 'American Express',
    'american water': 'American Water Works', 'american water works': 'American Water Works',
    'best buy': 'Best Buy',
    'boeing': 'Boeing',
    'cvs': 'CVS Health', 'cvs health': 'CVS Health',
    'coca-cola': 'Coca-Cola', 'coca cola': 'Coca-Cola', 'coke': 'Coca-Cola',
    'corning': 'Corning',
    'costco': 'Costco',
    'foot locker': 'Foot Locker',
    'general mills': 'General Mills',
    'jpmorgan': 'JPMorgan', 'jpm': 'JPMorgan', 'jp morgan': 'JPMorgan',
    'johnson': 'Johnson & Johnson', 'johnson & johnson': 'Johnson & Johnson', 'j&j': 'Johnson & Johnson',
    'lockheed': 'Lockheed Martin', 'lockheed martin': 'Lockheed Martin',
    'mgm': 'MGM Resorts', 'mgm resorts': 'MGM Resorts',
    'microsoft': 'Microsoft', 'msft': 'Microsoft',
    'nike': 'Nike',
    'netflix': 'Netflix', 'nflx': 'Netflix',
    'paypal': 'Paypal',
    'pepsico': 'PepsiCo', 'pepsi': 'PepsiCo',
    'pfizer': 'Pfizer',
    'ulta': 'Ulta Beauty', 'ulta beauty': 'Ulta Beauty',
    'verizon': 'Verizon',
    'walmart': 'Walmart',
    'block': 'Block', 'square': 'Block',
    'amazon': 'Amazon', 'amzn': 'Amazon',
    'kraft': 'Kraft Heinz', 'kraft heinz': 'Kraft Heinz',
}

def extract_company(question):
    q_lower = question.lower()
    for alias in sorted(COMPANY_ALIASES.keys(), key=len, reverse=True):
        if alias in q_lower:
            return COMPANY_ALIASES[alias]
    return None

def extract_year(question):
    fy_match = re.search(r'FY\s*(\d{4})', question, re.IGNORECASE)
    if fy_match:
        return fy_match.group(1)
    year_match = re.search(r'20\d{2}', question)
    if year_match:
        return year_match.group(0)
    return None

# ─── NEW: Financial Statement Type Detection ─────────────────

# Keywords in the QUESTION that hint at which statement is needed
STATEMENT_KEYWORDS = {
    'cash_flow': [
        'capital expenditure', 'capex', 'cash flow', 'cash provided',
        'purchases of property', 'dividends paid', 'free cash flow',
        'depreciation', 'amortization', 'investing activities',
        'financing activities', 'operating activities', 'fcf',
        'cash used', 'cash generated', 'repurchase',
    ],
    'income': [
        'revenue', 'sales', 'net income', 'operating income',
        'gross profit', 'earnings', 'eps', 'earnings per share',
        'operating margin', 'gross margin', 'profit margin',
        'cost of goods', 'cost of revenue', 'operating expense',
        'ebitda', 'ebit', 'income tax', 'net profit',
        'top line', 'bottom line', 'r&d', 'research and development',
        'sg&a', 'selling general',
    ],
    'balance': [
        'total assets', 'total liabilities', 'current assets',
        'current liabilities', 'equity', 'stockholders equity',
        'long-term debt', 'short-term debt', 'book value',
        'working capital', 'current ratio', 'quick ratio',
        'inventory', 'accounts receivable', 'accounts payable',
        'goodwill', 'intangible', 'net assets', 'ppe',
        'property plant', 'fixed asset',
    ],
}

# Markers found INSIDE the evidence text that identify statement type
STATEMENT_TEXT_MARKERS = {
    'cash_flow': [
        'cash flows from', 'cash provided by', 'investing activities',
        'financing activities', 'statement of cash flow',
        'cash used in', 'purchases of property',
    ],
    'income': [
        'net revenues', 'net sales', 'total revenues', 'cost of',
        'operating income', 'statements of operations',
        'statements of income', 'gross profit',
    ],
    'balance': [
        'total assets', 'total liabilities', 'stockholders equity',
        'balance sheet', 'current assets', 'current liabilities',
    ],
}

def detect_statement_type(question):
    """Detect which financial statement the question needs."""
    q_lower = question.lower()
    scores = {'cash_flow': 0, 'income': 0, 'balance': 0}
    for stmt_type, keywords in STATEMENT_KEYWORDS.items():
        for kw in keywords:
            if kw in q_lower:
                scores[stmt_type] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else None

def score_statement_match(evidence_text, target_statement):
    """Score how well an evidence passage matches the target statement type."""
    if not target_statement:
        return 0
    text_lower = evidence_text.lower()
    markers = STATEMENT_TEXT_MARKERS.get(target_statement, [])
    return sum(1 for m in markers if m in text_lower)

# ─── RAG Class with HARD filtering + Statement Type ──────────

class FinancialRAG:
    def __init__(self, embedding_model='all-MiniLM-L6-v2'):
        print(f"Loading embedding model: {embedding_model}")
        self.encoder = SentenceTransformer(embedding_model)
        self.evidence_store = []
        self.index = None
        self.all_embeddings = None

    def load_index(self, index_dir='./rag/vector_index'):
        self.index = faiss.read_index(f'{index_dir}/faiss_index.bin')
        with open(f'{index_dir}/evidence_store.pkl', 'rb') as f:
            self.evidence_store = pickle.load(f)

        self.all_embeddings = np.zeros((self.index.ntotal, self.index.d), dtype='float32')
        for i in range(self.index.ntotal):
            self.all_embeddings[i] = self.index.reconstruct(i)

        print(f"✅ Index loaded: {self.index.ntotal} vectors, {len(self.evidence_store)} chunks")
        print(f"✅ Embeddings extracted for filtered search")

    def _cosine_sim(self, query_emb, doc_embs):
        query_norm = query_emb / (np.linalg.norm(query_emb) + 1e-10)
        doc_norms = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-10)
        return np.dot(doc_norms, query_norm.T).flatten()

    def retrieve(self, question, k=3):
        question_embedding = self.encoder.encode([question], convert_to_numpy=True).astype('float32')
        target_company = extract_company(question)
        target_year = extract_year(question)
        target_statement = detect_statement_type(question)

        # ─── STEP 1: Hard filter — company + year ────────────
        filtered = []
        for i, (text, metadata) in enumerate(self.evidence_store):
            meta_company = metadata.get('company', '').lower()
            meta_doc = metadata.get('doc_name', '')
            company_match = target_company and target_company.lower() in meta_company
            year_match = target_year and target_year in meta_doc
            if company_match and year_match:
                filtered.append(i)

        # ─── STEP 2: If ZERO, relax to company only ──────────
        if len(filtered) == 0:
            for i, (text, metadata) in enumerate(self.evidence_store):
                meta_company = metadata.get('company', '').lower()
                if target_company and target_company.lower() in meta_company:
                    filtered.append(i)

        # ─── STEP 3: If still ZERO, search all ───────────────
        if len(filtered) == 0:
            filtered = list(range(len(self.evidence_store)))

        # ─── STEP 4: HYBRID scoring (Cosine + BM25) ──────────

        # Score A: Cosine similarity (semantic)
        filtered_embeddings = self.all_embeddings[filtered]
        cos_scores = self._cosine_sim(question_embedding[0], filtered_embeddings)

        if cos_scores.max() != cos_scores.min():
            cos_norm = (cos_scores - cos_scores.min()) / (cos_scores.max() - cos_scores.min())
        else:
            cos_norm = np.ones_like(cos_scores)

        # Score B: BM25 (keyword matching)
        filtered_texts = [self.evidence_store[i][0] for i in filtered]
        tokenized_docs = [doc.lower().split() for doc in filtered_texts]
        bm25 = BM25Okapi(tokenized_docs)
        bm25_scores = bm25.get_scores(question.lower().split())

        if bm25_scores.max() != bm25_scores.min():
            bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
        else:
            bm25_norm = np.zeros_like(bm25_scores)

        # ─── STEP 5: NEW — Statement type matching ───────────
        stmt_scores = np.zeros(len(filtered))
        if target_statement:
            for j, idx in enumerate(filtered):
                text = self.evidence_store[idx][0]
                stmt_scores[j] = score_statement_match(text, target_statement)
            if stmt_scores.max() > 0:
                stmt_norm = stmt_scores / stmt_scores.max()
            else:
                stmt_norm = np.zeros_like(stmt_scores)
        else:
            stmt_norm = np.zeros_like(stmt_scores)

        # ─── STEP 6: Combined score ──────────────────────────
        # 40% semantic + 30% keyword + 30% statement type
        combined_scores = 0.4 * cos_norm + 0.3 * bm25_norm + 0.3 * stmt_norm

        actual_k = min(k, len(filtered))
        top_indices = np.argsort(combined_scores)[::-1][:actual_k]

        candidates = []
        for idx in top_indices:
            real_idx = filtered[idx]
            text, metadata = self.evidence_store[real_idx]
            candidates.append({
                'evidence_text': text,
                'metadata': metadata,
                'distance': float(1 - combined_scores[idx]),
            })

        return candidates

# ─── Initialize ──────────────────────────────────────────────

rag = FinancialRAG()
rag.load_index('./rag/vector_index')

# ─── Test ────────────────────────────────────────────────────

print('\n' + '='*60)
print('METADATA + STATEMENT TYPE FILTERING TEST')
print('='*60)

test_qs = [
    "What is the FY2018 capital expenditure amount (in USD millions) for 3M?",
    "What is the FY2022 total revenue for Amazon?",
    "Does 3M have a reasonably healthy liquidity profile based on its FY2023 10Q?",
]

for q in test_qs:
    company = extract_company(q)
    year = extract_year(q)
    stmt = detect_statement_type(q)
    results = rag.retrieve(q, k=3)

    print(f'\nQ: {q[:70]}...')
    print(f'  Extracted → Company: {company}, Year: {year}, Statement: {stmt}')
    for j, r in enumerate(results):
        m = r['metadata']
        print(f'  [{j+1}] {m["company"]} - {m["doc_name"]} (score: {r["distance"]:.4f})')
        print(f'       Text preview: {r["evidence_text"][:80]}...')

print('\n✅ Cell 9a done!')

✅ RAG files already exist
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Index loaded: 189 vectors, 189 chunks
✅ Embeddings extracted for filtered search

METADATA + STATEMENT TYPE FILTERING TEST

Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M...
  Extracted → Company: 3M, Year: 2018, Statement: cash_flow
  [1] 3M - 3M_2018_10K (score: 0.4000)
       Text preview: Table of Contents 
3M Company and Subsidiaries
Consolidated Statement of Cash Fl...
  [2] 3M - 3M_2018_10K (score: 0.6000)
       Text preview: Table of Contents 
3M Company and Subsidiaries
Consolidated Balance Shee t
At De...

Q: What is the FY2022 total revenue for Amazon?...
  Extracted → Company: Amazon, Year: 2022, Statement: income
  [1] Amazon - AMAZON_2017_10K (score: 0.3000)
       Text preview: Table of Contents
AMAZON.COM, INC.
CONSOLIDATED STATEMENTS OF OPERATIONS
(in mil...
  [2] Amazon - AMAZON_2017_10K (score: 0.3000)
       Text preview: Table of Contents
AMAZON.COM, INC.
CONSOLIDATED STATEMENTS OF OPERATIONS
(in mil...
  [3] Amazon - AMAZON_2019_10K (

In [13]:
# =====================================================================
# CELL 9b: Llama 3.1 8B +  BM25 — Split by Question Type -> NEW WITH with hard metadata filtering.
# =====================================================================

# ─── Step 1: Categorize all questions ────────────────────────

def categorize_question(gold_answer):
    gold_clean = gold_answer.strip().lower()
    if gold_clean.startswith('yes') or gold_clean.startswith('no'):
        return 'yes_no'
    words = gold_answer.strip().split()
    if len(words) <= 4:
        nums = extract_all_numbers(gold_answer)
        if nums:
            return 'numerical'
    return 'descriptive'

for ex in qa:
    ex['category'] = categorize_question(ex['answer'])

cat_counts = {}
for ex in qa:
    cat_counts[ex['category']] = cat_counts.get(ex['category'], 0) + 1

print(f'Question Categories:')
print(f'  Numerical:   {cat_counts.get("numerical", 0)}')
print(f'  Yes/No:      {cat_counts.get("yes_no", 0)}')
print(f'  Descriptive: {cat_counts.get("descriptive", 0)}')
print(f'  Total:       {len(qa)}')

# ─── Step 2: Different prompts per category ──────────────────

def build_rag_prompt_numerical(question, evidence_text):
    return f"""You are a financial data extraction tool.
Extract ONLY the specific number or value asked for from the evidence below.
Do NOT explain. Do NOT repeat the question. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%")

Evidence:
{evidence_text}

Question: {question}

Answer (number only):"""

def build_rag_prompt_yesno(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Start with "Yes" or "No", then give a brief explanation with key numbers from the evidence.

Evidence:
{evidence_text}

Question: {question}

Answer (start with Yes or No):"""

def build_rag_prompt_descriptive(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Give a concise answer using specific numbers and facts from the evidence.
Cite the source document when possible.

Evidence:
{evidence_text}

Question: {question}

Answer:"""

# ─── Step 3: Run evaluation ─────────────────────────────────

print(f'\n{"=" * 70}')
print(f'Llama 3.1 8B +  BM25 — {len(qa)} Questions')
print(f'{"=" * 70}')

correct_rag = 0
results_rag = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence = ex.get('evidence', [])
    category = ex['category']

    # RAG: Retrieve evidence
    retrieved = rag.retrieve(q, k=3)
    evidence_text = '\n\n'.join([r['evidence_text'][:1500] for r in retrieved])

    # Pick the right prompt for this question type
    if category == 'numerical':
        prompt = build_rag_prompt_numerical(q, evidence_text)
    elif category == 'yes_no':
        prompt = build_rag_prompt_yesno(q, evidence_text)
    else:
        prompt = build_rag_prompt_descriptive(q, evidence_text)

    # Generate
    try:
        pred_raw = generate_answer(prompt)
        pred = extract_answer_from_verbose(pred_raw) if category == 'numerical' else pred_raw
    except Exception as e:
        print(f'  ⚠️ Error Q{i+1}: {e}')
        pred_raw = pred = 'ERROR'

    # Correctness check
    ok = is_correct(pred, gold)
    correct_rag += int(ok)

    # Store all info for split scoring later
    results_rag.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': pred_raw,
        'pred_answer': pred,
        'correct': ok,
        'category': category,
    })

    if (i + 1) % 10 == 0:
        acc_so_far = correct_rag / (i + 1) * 100
        print(f'  [{i+1:>3}/{len(qa)}] Accuracy: {acc_so_far:.1f}%')

    # Show first 2 of each category
    shown = {'numerical': 0, 'yes_no': 0, 'descriptive': 0}
    for r in results_rag:
        if r['category'] == category and shown[category] < 2:
            if i < 6:
                print(f'\n  [{category.upper()}] Q: {q[:65]}...')
                print(f'    Gold: {gold[:70]}')
                print(f'    Pred: {pred_raw[:70]}')
                print(f'    Correct: {"✅" if ok else "❌"}')
            shown[category] += 1

print(f'\n  Overall Accuracy: {correct_rag}/{len(qa)} ({correct_rag/len(qa)*100:.1f}%)')


# ─── Step 4: Split Scoring ───────────────────────────────────

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# NUMERICAL
num_r = [r for r in results_rag if r['category'] == 'numerical']
num_acc_list = [numerical_accuracy(r['pred_answer'], r['gold_answer']) for r in num_r]
num_hal_list = [numerical_hallucination_rate(r['pred_answer'], evidence_map.get(r['financebench_id'], [])) for r in num_r]
num_cit_list = [has_citation(r['pred_raw']) for r in num_r]

# YES/NO
yn_r = [r for r in results_rag if r['category'] == 'yes_no']
yn_correct = 0
yn_bleu_list = []
yn_sem_list = []
yn_cit_list = []
for r in yn_r:
    g = r['gold_answer'].strip().lower()
    p = r['pred_answer'].strip().lower()
    if (g.startswith('yes') and p.startswith('yes')) or (g.startswith('no') and p.startswith('no')):
        yn_correct += 1
    yn_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    yn_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    yn_cit_list.append(has_citation(r['pred_raw']))

# DESCRIPTIVE
desc_r = [r for r in results_rag if r['category'] == 'descriptive']
desc_bleu_list = []
desc_sem_list = []
desc_keynum_list = []
desc_cit_list = []
for r in desc_r:
    desc_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    desc_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    gold_nums = extract_all_numbers(r['gold_answer'])
    pred_nums = extract_all_numbers(r['pred_answer'])
    if gold_nums:
        matched = 0
        for g in gold_nums:
            for p in pred_nums:
                if g != 0 and abs(p - g) / abs(g) < 0.05:
                    matched += 1; break
                elif g == 0 and abs(p) < 1e-6:
                    matched += 1; break
        desc_keynum_list.append(matched / len(gold_nums))
    else:
        desc_keynum_list.append(1.0)
    desc_cit_list.append(has_citation(r['pred_raw']))


# ─── Step 5: Print Results ───────────────────────────────────

print(f'\n{"=" * 70}')
print(f'  Llama 3.1 8B +  BM25 — SPLIT RESULTS')
print(f'{"=" * 70}')

print(f'\n  📊 NUMERICAL ({len(num_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Numerical Accuracy:      {np.mean(num_acc_list)*100:.1f}%  ({int(sum(num_acc_list))}/{len(num_r)})')
print(f'  Numerical Hallucination: {np.mean(num_hal_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(num_cit_list)*100:.1f}%')

print(f'\n  ✅❌ YES/NO ({len(yn_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Yes/No Accuracy:         {yn_correct/len(yn_r)*100:.1f}%  ({yn_correct}/{len(yn_r)})')
print(f'  BLEU Score:              {np.mean(yn_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(yn_sem_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(yn_cit_list)*100:.1f}%')

print(f'\n  📝 DESCRIPTIVE ({len(desc_r)} questions)')
print(f'  {"─" * 50}')
print(f'  BLEU Score:              {np.mean(desc_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(desc_sem_list)*100:.1f}%')
print(f'  Key Number Capture:      {np.mean(desc_keynum_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(desc_cit_list)*100:.1f}%')

print(f'\n{"=" * 70}')

# Save
with open('results/llama_8b_rag_detailed_BM25.json', 'w') as f:
    json.dump(results_rag, f, indent=2)

with open('results/llama_8b_rag_split_metrics_BM25.json', 'w') as f:
    json.dump({
        'model': 'Llama-3.1-8B + RAG',
        'numerical': {
            'count': len(num_r),
            'accuracy': round(np.mean(num_acc_list)*100, 1),
            'hallucination': round(np.mean(num_hal_list)*100, 1),
            'citation': round(np.mean(num_cit_list)*100, 1),
        },
        'yes_no': {
            'count': len(yn_r),
            'yn_accuracy': round(yn_correct/len(yn_r)*100, 1),
            'bleu': round(np.mean(yn_bleu_list), 3),
            'semantic_similarity': round(np.mean(yn_sem_list)*100, 1),
            'citation': round(np.mean(yn_cit_list)*100, 1),
        },
        'descriptive': {
            'count': len(desc_r),
            'bleu': round(np.mean(desc_bleu_list), 3),
            'semantic_similarity': round(np.mean(desc_sem_list)*100, 1),
            'key_number_capture': round(np.mean(desc_keynum_list)*100, 1),
            'citation': round(np.mean(desc_cit_list)*100, 1),
        }
    }, f, indent=2)

print(f'\n✅ Cell 9b done!')
print(f'📌 Next: Run Cell 10 for BASE vs RAG comparison')

Question Categories:
  Numerical:   53
  Yes/No:      37
  Descriptive: 60
  Total:       150

Llama 3.1 8B +  BM25 — 150 Questions

  [NUMERICAL] Q: What is the FY2018 capital expenditure amount (in USD millions) f...
    Gold: $1577.00
    Pred: $1,577M
    Correct: ✅

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.7B
    Correct: ✅

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.7B
    Correct: ✅

  [YES_NO] Q: Is 3M a capital-intensive business based on FY2022 data?...
    Gold: No, the company is managing its CAPEX and Fixed Assets pretty efficien
    Pred: Yes
    Correct: ❌

  [DESCRIPTIVE] Q: What drove operating margin change as of FY2022 for 3M? If operat...
    Gold: Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due 
    Pred: Based on the provided evidence, the operating margin change for 3M as 
    Correct: ✅

  [DESCRI

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Llama 3.1 8B +  BM25 — SPLIT RESULTS

  📊 NUMERICAL (53 questions)
  ──────────────────────────────────────────────────
  Numerical Accuracy:      32.1%  (17/53)
  Numerical Hallucination: 58.5%
  Citation Rate:           0.0%

  ✅❌ YES/NO (37 questions)
  ──────────────────────────────────────────────────
  Yes/No Accuracy:         70.3%  (26/37)
  BLEU Score:              0.005
  Semantic Similarity:     17.3%
  Citation Rate:           0.0%

  📝 DESCRIPTIVE (60 questions)
  ──────────────────────────────────────────────────
  BLEU Score:              0.061
  Semantic Similarity:     60.1%
  Key Number Capture:      62.1%
  Citation Rate:           81.7%


✅ Cell 9b done!
📌 Next: Run Cell 10 for BASE vs RAG comparison


In [14]:
# =====================================================================
# VERIFY: Check the 17 correct numerical answers
# =====================================================================

correct_num = [r for r in results_rag if r['category'] == 'numerical' and r['correct']]
wrong_num = [r for r in results_rag if r['category'] == 'numerical' and not r['correct']]

print(f"✅ CORRECT NUMERICAL: {len(correct_num)}/53")
print(f"{'='*70}")
for i, r in enumerate(correct_num):
    print(f"  [{i+1}] {r['company']} - {r['doc_name']}")
    print(f"      Q: {r['question'][:70]}...")
    print(f"      Gold: {r['gold_answer']}")
    print(f"      Pred: {r['pred_answer']}")
    print()

print(f"\n❌ WRONG NUMERICAL (first 10): {len(wrong_num)}/53")
print(f"{'='*70}")
for i, r in enumerate(wrong_num[:10]):
    retrieved = rag.retrieve(r['question'], k=3)
    stmt = detect_statement_type(r['question'])
    print(f"  [{i+1}] {r['company']} - {r['doc_name']}")
    print(f"      Q: {r['question'][:70]}...")
    print(f"      Gold: {r['gold_answer']}")
    print(f"      Pred: {r['pred_answer']}")
    print(f"      Statement detected: {stmt}")
    print(f"      Retrieved: {[ret['metadata']['doc_name'][:30] for ret in retrieved]}")
    print()

✅ CORRECT NUMERICAL: 17/53
  [1] 3M - 3M_2018_10K
      Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M...
      Gold: $1577.00
      Pred: $1,577M

  [2] 3M - 3M_2018_10K
      Q: Assume that you are a public equities analyst. Answer the following qu...
      Gold: $8.70
      Pred: $8.7B

  [3] AES Corporation - AES_2022_10K
      Q: What is the quantity of restructuring costs directly outlined in AES C...
      Gold: 0
      Pred: 0

  [4] Amazon - AMAZON_2019_10K
      Q: By drawing conclusions from the information stated only in the income ...
      Gold: $11588.00
      Pred: $11,588.0M

  [5] Amcor - AMCOR_2020_10K
      Q: What is Amcor's year end FY2020 net AR (in USD millions)? Address the ...
      Gold: $1616.00
      Pred: 1,615.9

  [6] Best Buy - BESTBUY_2019_10K
      Q: What is the year end FY2019 total amount of inventories for Best Buy? ...
      Gold: $5409.00
      Pred: 5,409.0M

  [7] Block - BLOCK_2020_10K
      Q: Using the cash flow s

In [15]:
# =====================================================================
# VERIFY: Check Yes/No answers
# =====================================================================

correct_yn = [r for r in results_rag if r['category'] == 'yes_no' and r['correct']]
wrong_yn = [r for r in results_rag if r['category'] == 'yes_no' and not r['correct']]

print(f"✅ CORRECT YES/NO: {len(correct_yn)}/37")
print(f"{'='*70}")
for i, r in enumerate(correct_yn):
    gold_dir = "YES" if r['gold_answer'].strip().lower().startswith('yes') else "NO"
    pred_dir = r['pred_answer'].strip()[:30]
    print(f"  [{i+1}] {r['company']}")
    print(f"      Q: {r['question'][:65]}...")
    print(f"      Gold direction: {gold_dir}")
    print(f"      Pred: {pred_dir}")
    print()

print(f"\n❌ WRONG YES/NO: {len(wrong_yn)}/37")
print(f"{'='*70}")
for i, r in enumerate(wrong_yn):
    gold_dir = "YES" if r['gold_answer'].strip().lower().startswith('yes') else "NO"
    pred_dir = r['pred_answer'].strip()[:50]
    print(f"  [{i+1}] {r['company']}")
    print(f"      Q: {r['question'][:65]}...")
    print(f"      Gold: {gold_dir} — {r['gold_answer'][:60]}")
    print(f"      Pred: {pred_dir}")
    print()

✅ CORRECT YES/NO: 1/37
  [1] American Express
      Q: Was American Express able to retain card members during 2022?...
      Gold direction: YES
      Pred: Yes


❌ WRONG YES/NO: 36/37
  [1] 3M
      Q: Is 3M a capital-intensive business based on FY2022 data?...
      Gold: NO — No, the company is managing its CAPEX and Fixed Assets prett
      Pred: Yes

  [2] 3M
      Q: Does 3M have a reasonably healthy liquidity profile based on its ...
      Gold: NO — No. The quick ratio for 3M was 0.96 by Jun'23 close, which n
      Pred: Yes

  [3] 3M
      Q: Does 3M maintain a stable trend of dividend distribution?...
      Gold: YES — Yes, not only they distribute the dividends on a routine bas
      Pred: Yes

  [4] Adobe
      Q: Does Adobe have an improving operating margin profile as of FY202...
      Gold: NO — No the operating margins of Adobe have recently declined fro
      Pred: Yes

  [5] Adobe
      Q: Does Adobe have an improving Free cashflow conversion as of FY202...
      Gol

In [16]:
# Show what the model ACTUALLY generated for a few Yes/No questions
yn_samples = [r for r in results_rag if r['category'] == 'yes_no'][:5]

for r in yn_samples:
    print(f"Q: {r['question'][:70]}...")
    print(f"Gold: {r['gold_answer'][:60]}")
    print(f"Model raw output: {r['pred_raw'][:150]}")
    print(f"---")

Q: Is 3M a capital-intensive business based on FY2022 data?...
Gold: No, the company is managing its CAPEX and Fixed Assets prett
Model raw output: Yes
---
Q: Does 3M have a reasonably healthy liquidity profile based on its quick...
Gold: No. The quick ratio for 3M was 0.96 by Jun'23 close, which n
Model raw output: Yes
---
Q: Does 3M maintain a stable trend of dividend distribution?...
Gold: Yes, not only they distribute the dividends on a routine bas
Model raw output: Yes
---
Q: Does Adobe have an improving operating margin profile as of FY2022? If...
Gold: No the operating margins of Adobe have recently declined fro
Model raw output: Yes
---
Q: Does Adobe have an improving Free cashflow conversion as of FY2022?...
Gold: Yes, the FCF conversion (using net income as the denominator
Model raw output: Yes
---
